In [1]:
from datasets import load_dataset

ds = load_dataset("urish/pirate-chat", split="train")
ds[0]

{'user': "What's the weather like today?",
 'answer': "Ahoy matey! The skies be lookin' mighty fine today, with fair winds blowin' and nary a storm cloud in sight!",
 'source': 'claude'}

In [2]:
import tinker

MODEL_ID = "Qwen/Qwen3.5-4B"

service_client = tinker.ServiceClient()
training_client = service_client.create_lora_training_client(
    base_model=MODEL_ID, rank=16,
)
tokenizer = training_client.get_tokenizer()

In [3]:
from tinker.types import ModelInput, Datum, TensorData
import torch

with open("chat_template.txt") as f:
    chat_template = f.read()

data = []
for row in ds:
    messages = [
        {"role": "user", "content": row["user"]},
        {"role": "assistant", "content": row["answer"]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        chat_template=chat_template,
        return_assistant_tokens_mask=True,
        enable_thinking=False,
    )
    ids = inputs["input_ids"]
    weights = inputs["assistant_masks"]

    data.append(Datum(
        model_input=ModelInput.from_ints(ids[:-1]),
        loss_fn_inputs={
            "target_tokens": TensorData.from_torch(torch.tensor(ids[1:])),
            "weights": TensorData.from_torch(torch.tensor(weights[1:], dtype=torch.float32)),
        },
    ))

In [ ]:
from tinker.types import AdamParams
from tqdm import tqdm
import math

def lr(w, min_lr, max_lr):
    return min_lr + (0.5 + math.cos(math.pi * w) * 0.5) * (max_lr - min_lr)

with tqdm(range(0, len(data), 16)) as pbar:
    for i in pbar:
        fwd_bwd_future = await training_client.forward_backward_async(
            data=data[i:i + 16], loss_fn="cross_entropy",
        )
        optim_future = await training_client.optim_step_async(
            AdamParams(learning_rate=lr(i / len(data), min_lr=1e-5, max_lr=2e-4)),
        )
        fwd_bwd = await fwd_bwd_future.result_async()
        await optim_future.result_async()
        pbar.set_postfix(**fwd_bwd.metrics)

100%|██████████| 72/72 [04:39<00:00,  3.88s/it, clock_cycle:unique=5.83e+6, loss:sum=926]    


In [5]:
from tinker.types import SamplingParams

sampling_client = await training_client.save_weights_and_get_sampling_client_async()

messages = [{"role": "user", "content": "Tell me about yourself."}]
inputs = tokenizer.apply_chat_template(
    messages,
    chat_template=chat_template,
    add_generation_prompt=True,
    enable_thinking=False,
)
ids = inputs["input_ids"]

result = await sampling_client.sample_async(
    prompt=ModelInput.from_ints(ids),
    num_samples=1,
    sampling_params=SamplingParams(max_tokens=128, temperature=0.7),
)
print(tokenizer.decode(result.sequences[0].tokens))

I be a landlubber, a proper landlubber, born in a port, raised in a port, and a landlubber to the end, matey!<|im_end|>


Very piratey. Nice.